# Coverage Alpha Probe

用 sum (不除 n_posts) + Ridge(alpha=100) 固定 + winsorize + 跳过 <20 天 fold。

In [4]:
# 1. 加载 + 投影 + 收益 + walk-forward + h_cache
import os, sys, time, glob, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
os.chdir(ROOT); sys.path.insert(0, ROOT)
# --- 从 per_file/ 加载 (用 sum, 不除以 n_posts) ---
per_dir = os.path.join(ROOT, "artifacts", "gubapost_cls", "per_file")
pf_files = sorted(glob.glob(os.path.join(per_dir, "*.parquet")))
assert pf_files, f"per_file/ 下没有文件 -> 先跑 extract_gubapost_cls.ipynb"
print(f"加载 {len(pf_files)} 个 per-file parquet...")
dfs = [pd.read_parquet(f) for f in pf_files]
pf = pd.concat(dfs, ignore_index=True); del dfs
cls = np.stack(pf["sum_cls"].values).astype(np.float32)  # sum, 不是 mean
cls_df = pf[["available_date", "symbol", "n_posts"]].copy()
print(f"CLS(sum): {cls.shape} | dates: {cls_df.available_date.min()}~{cls_df.available_date.max()}")
# --- 收益 (winsorize, 前移1天) ---
rtn = pd.read_parquet("/home/intern_fjq_2026/data/RTN_daily/rtn_1d.parquet")
rtn_long = rtn.melt(id_vars="date", var_name="sym", value_name="r")
rtn_long["symbol"] = rtn_long["sym"].str.split(".").str[0]
rtn_long["date"] = pd.to_datetime(rtn_long["date"]).dt.strftime("%Y-%m-%d")
rtn_long = rtn_long.sort_values(["symbol", "date"])
rtn_long["y"] = rtn_long.groupby("symbol")["r"].shift(-1)
rtn_long = rtn_long[["date", "symbol", "y"]].dropna(subset=["y"])
rtn_long["y"] = rtn_long["y"].clip(-0.2, 0.2)
merged = cls_df[["available_date", "symbol"]].merge(
    rtn_long, left_on=["available_date", "symbol"], right_on=["date", "symbol"], how="inner")
merged = merged[["available_date", "symbol", "y"]].reset_index(drop=True)
merged["year"] = merged["available_date"].str[:4]

# 只用已完成的年份
PROBE_YEARS = {"2020", "2021"}
years = sorted(merged["year"].unique())
MIN_TEST_DAYS = 20
folds = [(years[:i], years[i]) for i in range(1, len(years))
         if (merged["year"] == years[i]).sum() >= MIN_TEST_DAYS]
print(f"walk-forward: {len(folds)} folds")
idx_all = np.arange(len(merged))
y_all = merged["y"].values.astype(np.float64)
dates_all = merged["available_date"].values
from sklearn.linear_model import Ridge
ALPHA = 100
def ric(yp, yt, d):
    ics = []
    for dt in np.unique(d):
        m = d == dt
        if m.sum() >= 10: ics.append(spearmanr(yp[m], yt[m])[0])
    return np.array(ics)
def rfp(Xtr, ytr, Xte):
    sc = StandardScaler()
    m = Ridge(alpha=ALPHA)
    m.fit(sc.fit_transform(Xtr), ytr)
    return m.predict(sc.transform(Xte))
def safe_icir(a):
    s = a.std()
    return a.mean()/s if s > 1e-6 else 0
# --- H 方向 + h_cache ---
DIRS_PATH = os.path.join(ROOT, "artifacts", "checkpoint_activation_rank", "runs", "gubapost_v1",
                         "extensions", "coverage_ablation_v1", "direction_sets.npz")
dirs_npz = np.load(DIRS_PATH)
H = cls @ dirs_npz["keep_317_complement_K64"].astype(np.float32)
L_cov = cls @ dirs_npz["keep_451_lowcov_K64"].astype(np.float32)
HL = np.hstack([H, L_cov])
print(f"H {H.shape} L {L_cov.shape}")
h_cache = {}
for fi, (tr, te) in enumerate(folds):
    ti = idx_all[merged["year"].isin(tr).values]
    ei = idx_all[(merged["year"] == te).values]
    kf = KFold(5, shuffle=False)
    oos = np.full(len(ti), np.nan)
    for a, b in kf.split(H[ti]):
        sc = StandardScaler(); m = Ridge(alpha=ALPHA)
        m.fit(sc.fit_transform(H[ti][a]), y_all[ti][a])
        oos[b] = m.predict(sc.transform(H[ti][b]))
    train_res = y_all[ti] - oos
    ypH = rfp(H[ti], y_all[ti], H[ei])
    h_cache[fi] = (train_res, y_all[ei] - ypH, dates_all[ei], ti, ei)
print("h_cache 完成")

加载 914 个 per-file parquet...
CLS(sum): (8683669, 768) | dates: 2020-01-02~2026-06-03
walk-forward: 5 folds
H (8683669, 317) L (8683669, 451)


/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.6945e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.78664e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.43317e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.22552e-08): result may not be accurate.
  return linal

h_cache 完成


In [ ]:
# 2. Ridge probe + Residual IC + Random baseline
probe = {}
for name, feat in [("H_317", H), ("L_451", L_cov), ("HL_768", HL)]:
    probe[name] = []
    for tr, te in folds:
        ti = idx_all[merged["year"].isin(tr).values]
        ei = idx_all[(merged["year"] == te).values]
        yp = rfp(feat[ti], y_all[ti], feat[ei])
        ics = ric(yp, y_all[ei], dates_all[ei])
        probe[name].append((te, ics))
        print(f"  {name} {te}: IC={ics.mean():.4f} ICIR={safe_icir(ics):.2f}")

ic_H = np.concatenate([r[1] for r in probe["H_317"]]).mean()
ic_L = np.concatenate([r[1] for r in probe["L_451"]]).mean()
ic_HL = np.concatenate([r[1] for r in probe["HL_768"]]).mean()
print(f"\nIC_H={ic_H:.4f} IC_L={ic_L:.4f} IC_{{H+L}}={ic_HL:.4f} dIC={ic_HL-ic_H:.4f}")

# Residual IC
res_ics = []
for fi, (tr, te) in enumerate(folds):
    train_res, test_res, test_dates, ti, ei = h_cache[fi]
    rp = rfp(L_cov[ti], train_res, L_cov[ei])
    ics = ric(rp, test_res, test_dates)
    res_ics.append((te, ics))
    print(f"  ResIC {te}: {ics.mean():.4f}")
all_res = np.concatenate([r[1] for r in res_ics])
res_mean = all_res.mean()
print(f"ResidualIC_{{L|H}} = {res_mean:.4f}")

# Random baseline (50, fixed alpha)
rand_ics = []
for seed in range(50):
    rng = np.random.default_rng(seed)
    Qr = np.linalg.qr(rng.standard_normal((768, 451)))[0].astype(np.float32)
    R = cls @ Qr
    fm = []
    for fi, (tr, te) in enumerate(folds):
        train_res, test_res, test_dates, ti, ei = h_cache[fi]
        rp = rfp(R[ti], train_res, R[ei])
        fm.append(ric(rp, test_res, test_dates).mean())
    rand_ics.append(np.mean(fm))
    if (seed+1) % 10 == 0: print(f"  random {seed+1}/50: {np.mean(fm):.4f}")
rand_m, rand_s = np.mean(rand_ics), np.std(rand_ics)
print(f"\nRandom: {rand_m:.4f} +/- {rand_s:.4f} (2sigma={rand_m+2*rand_s:.4f})")
print(f"L vs random: {'SIGNIFICANT' if res_mean > rand_m+2*rand_s else 'marginal' if res_mean > rand_m else 'not significant'}")

  H_317 2021: IC=0.0002 ICIR=0.01
  H_317 2022: IC=0.0158 ICIR=0.00
  H_317 2023: IC=0.0008 ICIR=0.05


/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.43317e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  H_317 2024: IC=0.0013 ICIR=0.10


/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.24874e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  H_317 2025: IC=-0.0004 ICIR=-0.04
  L_451 2021: IC=0.0006 ICIR=0.03


...

/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.16865e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.37013e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.57484e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.89376e-08): result may not be accurate.
  return lina

  random 20/50: -0.0060


/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.09629e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.0495e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.35517e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=3.90057e-08): result may not be accurate.
  return linal

  random 30/50: -0.0047


/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.68379e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=5.70581e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.92271e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/home/intern_fjq_2026/miniconda3/envs/nlp_fjq/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:200: LinAlgWarning: Ill-conditioned matrix (rcond=4.72032e-08): result may not be accurate.
  return lina

In [ ]:
# 3. 汇总 + 保存 + 图
import matplotlib.pyplot as plt
rows = []
for nm in ["H_317", "L_451", "HL_768"]:
    a = np.concatenate([r[1] for r in probe[nm]])
    rows.append({"statistic":f"IC_{nm}","mean_IC":a.mean(),"ICIR":safe_icir(a),"t_stat":safe_icir(a)*np.sqrt(len(a))})
rows.append({"statistic":"dIC_joint","mean_IC":ic_HL-ic_H,"ICIR":np.nan,"t_stat":np.nan})
rows.append({"statistic":"ResidualIC_{L|H}","mean_IC":res_mean,"ICIR":safe_icir(all_res),"t_stat":safe_icir(all_res)*np.sqrt(len(all_res))})
rows.append({"statistic":"Random(mean)","mean_IC":rand_m,"ICIR":np.nan,"t_stat":np.nan})
rows.append({"statistic":"Random(std)","mean_IC":rand_s,"ICIR":np.nan,"t_stat":np.nan})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_parquet(os.path.join(ROOT,"artifacts","gubapost_cls","coverage_alpha_probe_results.parquet"), index=False)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(["IC_H","IC_L","IC_{H+L}","ResIC\n{L|H}","Random"],
       [ic_H, ic_L, ic_HL, res_mean, rand_m],
       yerr=[0,0,0,0,rand_s], color=["blue","orange","green","red","gray"], capsize=5)
ax.axhline(0,color="black",lw=0.5); ax.set_ylabel("mean daily Rank IC"); ax.set_title("Coverage Alpha Probe")
plt.tight_layout(); plt.show()